# Epi Info AI TABLES validation lab — V0.6

Independently validate the browser candidate for `TABLES potato_salad case_status STRATAVAR=Sex` with Python and SciPy. This notebook validates the TypeScript result contract; TABLES is not yet a Rust/WASM kernel and passing is not legacy parity approval.

In [ ]:
import csv, hashlib, io, math
from collections import Counter
from pyodide.http import pyfetch
fixtures = []
for fixture_name in ('foodborne-tables-stratified-v0.3.json', 'foodborne-tables-unstratified-v0.3.json'):
    fixture_response = await pyfetch('../../validation-fixtures/' + fixture_name)
    fixture_response.raise_for_status(); fixtures.append(await fixture_response.json())
data_response = await pyfetch('../../examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert all(hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256'] for fixture in fixtures)
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
assert all(len(records) == fixture['dataset']['rows'] == 96 for fixture in fixtures)


In [ ]:
for fixture in fixtures:
    request = fixture['request']
    selected_headers = [request['exposureHeader'], request['outcomeHeader']] + ([request['strataHeader']] if 'strataHeader' in request else [])
    included = [r for r in records if all(r[h].strip() for h in selected_headers)]
    assert len(included) == fixture['includedRecords']
    assert len(records) - len(included) == fixture['excludedMissing']
    for expected_stratum in fixture['strata']:
        members = [r for r in included if 'strataHeader' not in request or r[request['strataHeader']] == expected_stratum['value']]
        counts = Counter((r[request['exposureHeader']], r[request['outcomeHeader']]) for r in members)
        matrix = [[counts[(exposure, outcome)] for outcome in fixture['outcomeValues']] for exposure in fixture['exposureValues']]
        assert matrix == [row['counts'] for row in expected_stratum['rows']]
print('PASS: canonical foodborne records reproduce the unstratified and stratified 2 × 4 count matrices')


In [ ]:
import numpy as np
from scipy.stats import chi2_contingency
for fixture in fixtures:
    for expected_stratum in fixture['strata']:
        observed = np.array([row['counts'] for row in expected_stratum['rows']], dtype=float)
        chi_square, p_value, df, expected = chi2_contingency(observed, correction=False)
        candidate = expected_stratum['pearson']
        assert math.isclose(chi_square, candidate['chiSquare'], abs_tol=1e-12, rel_tol=0)
        assert df == candidate['degreesOfFreedom']
        assert math.isclose(p_value, candidate['pValue'], abs_tol=1e-12, rel_tol=0)
        assert np.allclose(expected, np.array([row['expectedCounts'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        row_percent = observed / observed.sum(axis=1, keepdims=True) * 100
        column_percent = observed / observed.sum(axis=0, keepdims=True) * 100
        assert np.allclose(row_percent, np.array([row['rowPercents'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        assert np.allclose(column_percent, np.array([row['columnPercents'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        assert candidate['cellsExpectedBelowFive'] == int((expected < 5).sum())
        assert candidate['cellsExpectedBelowOne'] == int((expected < 1).sum())
print('PASS: independent SciPy M×N chi-square, expected counts, percentages, and sparse-cell diagnostics match V0.6')


In [ ]:
# Independent Fisher-Freeman-Halton enumeration; this does not call the TypeScript candidate.
import itertools
fisher_response = await pyfetch('../../validation-fixtures/foodborne-tables-fisher-v0.5.json')
fisher_response.raise_for_status(); fisher_fixture = await fisher_response.json()
observed = fisher_fixture['expected']['counts']; margins = [sum(column) for column in zip(*observed)]
row_total = sum(observed[0]); denominator = math.comb(sum(margins), row_total)
def probability(top):
    return math.prod(math.comb(margin, value) for margin, value in zip(margins, top)) / denominator
observed_probability = probability(observed[0]); probabilities = []
for prefix in itertools.product(*(range(margin + 1) for margin in margins[:-1])):
    final = row_total - sum(prefix)
    if 0 <= final <= margins[-1]: probabilities.append(probability((*prefix, final)))
tolerance = fisher_fixture['request']['tolerance']
two_tailed = sum(value for value in probabilities if value <= observed_probability * (1 + tolerance))
assert len(probabilities) == fisher_fixture['expected']['tablesEnumerated'] == 2737
assert math.isclose(two_tailed, fisher_fixture['expected']['twoTailedPValue'], abs_tol=1e-25, rel_tol=0)
print('PASS: independent Python Fisher-Freeman-Halton enumeration matches TABLES V0.6')


In [ ]:
missing_response = await pyfetch('../../validation-fixtures/foodborne-tables-missing-v0.6.json')
missing_response.raise_for_status(); missing_fixture = await missing_response.json()
blank = [record for record in records if not record['Vomiting'].strip() or not record['Sex'].strip()]
assert len(blank) == missing_fixture['expected']['includedMissing'] == 2
missing_counts = Counter(record['Sex'] for record in blank)
assert [missing_counts['Female'], missing_counts['Male']] == missing_fixture['expected']['missingExposureCounts'] == [0, 2]
assert missing_fixture['request']['source'].splitlines() == ['SET (.)="Not recorded"', 'SET MISSING=ON', 'TABLES vomiting Sex', 'SET MISSING=OFF', 'SET (.)="Missing"']
print('PASS: foodborne blanks independently confirm the TABLES V0.6 SET MISSING=ON/OFF fixture')
